# 03 · Ripeness & ML Model Ops
Probe ripeness scoring, fetch training history, run smoke evaluations, and
inspect ensemble candidates.

**Covered endpoints**
- `POST /ripeness` — score ripeness for a sample
- `GET  /ripeness/summary` — aggregate ripeness stats
- `GET  /banana-ml/training/history` — training run history
- `POST /banana-ml/training/smoke` — left-brain CI smoke
- `GET  /banana-ml/candidate` — current candidate model info
- `POST /banana-ml/candidate/promote` — promote candidate to production

In [ ]:
import json, urllib.request, urllib.error
API_BASE = 'https://api.banana.engineer'

def pp(obj):
    if isinstance(obj, str):
        try: obj = json.loads(obj)
        except Exception: print(obj); return
    print(json.dumps(obj, indent=2))

def call_endpoint(method, path, payload=None, headers=None):
    url = f"{API_BASE}{path}"
    h = {'Accept': 'application/json'}
    if headers: h.update(headers)
    data = None
    if payload is not None:
        h['Content-Type'] = 'application/json'
        data = json.dumps(payload).encode()
    req = urllib.request.Request(url, data=data, headers=h, method=method.upper())
    try:
        with urllib.request.urlopen(req) as r:
            return {'ok': True, 'status': r.status, 'body': r.read().decode('utf-8', errors='replace')}
    except urllib.error.HTTPError as e:
        return {'ok': False, 'status': e.code, 'body': e.read().decode('utf-8', errors='replace')}
    except Exception as ex:
        return {'ok': False, 'status': None, 'body': str(ex)}

def get(path, **kw):  return call_endpoint('GET',  path, **kw)
def post(path, **kw): return call_endpoint('POST', path, **kw)
print('setup_ok')

In [ ]:
# ── 1. Ripeness summary ───────────────────────────────────────────────────
r = get('/ripeness/summary')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 2. Single ripeness score probe ───────────────────────────────────────
# Adjust payload fields to match your RipenessRequest schema
payload = {'purchases': 4, 'multiplier': 2}
r = post('/ripeness', payload=payload)
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 3. Ripeness sweep — multiplier 1-10 ──────────────────────────────────
sweep_results = []
for m in range(1, 11):
    r = post('/ripeness', payload={'purchases': 3, 'multiplier': m})
    try:
        body = json.loads(r['body'])
    except Exception:
        body = r['body']
    sweep_results.append({'multiplier': m, 'status': r['status'], 'body': body})

print('sweep_done')
for row in sweep_results:
    print(row)

In [ ]:
# ── 4. Training history ───────────────────────────────────────────────────
r = get('/banana-ml/training/history')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 5. Left-brain CI smoke ────────────────────────────────────────────────
# Dry-run: does not promote; validates model pipeline integrity
r = post('/banana-ml/training/smoke')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 6. Current candidate model ────────────────────────────────────────────
r = get('/banana-ml/candidate')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 7. Promote candidate  [GUARDED — uncomment to execute] ───────────────
# WARNING: This promotes the current candidate to production.
# Only run after reviewing candidate metrics above.
#
# r = post('/banana-ml/candidate/promote')
# print('status:', r['status'])
# pp(r['body'])